# v4 Operator Learner 训练

## Goal

加载 v4 FE 和归一化参数，训练 Operator Learner。每个训练样本继续独立采样
$(\sigma_{\theta},\sigma_r)$；训练时同时保存随机分布验证和解析真解预测历史。


## Setup

### Key Assumptions

- 已完成同一 `run_name` 的 v4 FE 训练。
- `config_fe.json`、`fe_params.msgpack` 和归一化文件位于运行目录。
- 解析监控采用 $f=0$、$g_n=\cos{\theta}$、$k=1$，只用于验证。


In [ ]:
from dataclasses import fields, replace
from pathlib import Path
import json
import sys

import jax
import matplotlib.pyplot as plt
import numpy as np
from flax import serialization

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "config_polar.py").exists():
    PROJECT_DIR = Path("/path/to/polar_annulus_sno_code_v4")
if not (PROJECT_DIR / "config_polar.py").exists():
    raise FileNotFoundError("请将 PROJECT_DIR 改为 v4 代码目录。")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from config_polar import PolarAnnulusConfig
from train_polar import (
    create_fe_state,
    load_normalizer,
    train_operator,
)

print("Project directory:", PROJECT_DIR)
print("JAX devices:", jax.devices())


## Steps

### 1. Restore the FE configuration


In [ ]:
RUN_NAME = "polar_v4"
OUTPUT_ROOT = PROJECT_DIR / "out_polar_annulus_sno_v4"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CONFIG_PATH = RUN_DIR / "config_fe.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"找不到 FE 配置：{CONFIG_PATH}")

saved = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
valid_fields = {field.name for field in fields(PolarAnnulusConfig)}
kwargs = {key: value for key, value in saved.items() if key in valid_fields}
for key in (
    "sigma_theta_range",
    "sigma_r_range",
    "cnn_channels",
    "cnn_kernel_size",
    "cnn_stride",
):
    if key in kwargs:
        kwargs[key] = tuple(kwargs[key])
kwargs["out_dir"] = str(OUTPUT_ROOT)
kwargs["run_name"] = RUN_NAME
fe_cfg = PolarAnnulusConfig(**kwargs)

print("FE run directory:", fe_cfg.output_dir)
print("FE sigma ranges:", fe_cfg.sigma_theta_range, fe_cfg.sigma_r_range)


### 2. Configure OL without changing the network architecture


In [ ]:
OL_RUN_MODE = "full"  # "smoke" or "full"

if OL_RUN_MODE == "smoke":
    ol_cfg = replace(
        fe_cfg,
        sample_size=2,
        prior_generation_chunk_size=1,
        ol_steps=2,
        ol_log_interval=1,
        ol_checkpoint_interval=2,
        ol_eval_sample_size=2,
        ol_eval_probe_points=8,
        ol_exact_eval_interval=1,
        exact_eval_radial_size=4,
        exact_eval_theta_size=8,
        exact_eval_save_figure=False,
    )
else:
    # v3 OL used 3 x 128 = 384 samples per step. v4 keeps the same
    # total batch size while replacing the three fixed groups by
    # 384 independently sampled scale pairs.
    ol_cfg = replace(
        fe_cfg,
        sample_size=384,
        prior_generation_chunk_size=128,
        ol_steps=300_000,
        ol_log_interval=500,
        ol_checkpoint_interval=10_000,
        ol_eval_sample_size=16,
        ol_eval_probe_points=256,
        ol_exact_eval_interval=5_000,
    )

print("OL total batch size:", ol_cfg.effective_batch_size)
print("OL steps:", ol_cfg.ol_steps)
print("Exact monitor every:", ol_cfg.ol_exact_eval_interval)


### 3. Load the frozen FE and normalizer


In [ ]:
fe_checkpoint = RUN_DIR / "fe_params.msgpack"
if not fe_checkpoint.exists():
    raise FileNotFoundError(f"找不到 FE checkpoint：{fe_checkpoint}")

fe_state, _ = create_fe_state(fe_cfg, jax.random.PRNGKey(fe_cfg.seed + 100))
fe_params = serialization.from_bytes(
    fe_state.params, fe_checkpoint.read_bytes()
)
fe_state_for_ol = fe_state.replace(params=fe_params)
normalizer_for_ol = load_normalizer(RUN_DIR)

print("Loaded FE:", fe_checkpoint)
print("Normalizer std P:", float(normalizer_for_ol.std_p))
print("Normalizer std f:", float(normalizer_for_ol.std_f))


### 4. Train OL

输出包括 `operator_training_history.*`、`operator_exact_history.*`、checkpoint 和 `exact_monitor/ol_step_*.png`。


In [ ]:
ol_state = train_operator(
    ol_cfg,
    fe_state_for_ol,
    normalizer_for_ol,
)


## Checks

### 5. Plot random and analytic prediction histories


In [ ]:
def load_history(path):
    if not path.exists():
        print("Not found:", path)
        return None
    return np.atleast_1d(np.genfromtxt(path, delimiter=",", names=True))


random_history = load_history(RUN_DIR / "operator_training_history.csv")
exact_history = load_history(RUN_DIR / "operator_exact_history.csv")

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.4), layout="constrained")
if random_history is not None:
    axes[0].semilogy(
        random_history["step"],
        random_history["eval_latent_relative_l2"],
        label="latent RL2",
    )
    axes[0].semilogy(
        random_history["step"],
        random_history["eval_physical_relative_l2"],
        label="physical RL2",
    )
if exact_history is not None:
    axes[1].semilogy(
        exact_history["step"],
        exact_history["p_area_relative_l2"],
        label="exact P, area weighted",
    )
    axes[1].semilogy(
        exact_history["step"],
        exact_history["inner_flux_relative_l2"],
        label="inner flux RL2",
    )
for axis in axes:
    axis.set_xlabel("Step")
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
axes[0].set_title("Fresh random validation")
axes[1].set_title("Fixed analytic monitor")
plt.show()


## Next Steps

训练完成后使用未参与调参的 Fourier 模态、相位或 $k$ 组合做最终解析测试。反复查看的固定解析曲线只能作为训练监控，不能单独证明整个先验矩形上的泛化能力。
